# V13a: ConvNeXt-V2-Base + Strong Color Jitter

v8 + 색상/밝기 augmentation 극강화 (색상 의존 차단 → 형태에 집중)

**변경**: Color Jitter, Brightness, Contrast 파라미터만 강화. 나머지 v8 그대로.


In [1]:
# === Section 0: Imports + Config ===
import copy
import gc
import json
import math
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from torch.amp import autocast
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

warnings.filterwarnings('ignore')


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))


@dataclass
class Config:
    # === v8: ConvNeXt-V2-Base ===
    backbone: str = 'convnextv2_base.fcmae_ft_in22k_in1k'
    exp_name: str = 'v13a_strong_jitter'
    img_size: int = 384
    epochs: int = 15
    batch_size: int = 4
    lr: float = 1e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 2
    early_stopping_patience: int = 7
    grad_clip: float = 1.0
    use_ema: bool = True
    ema_decay: float = 0.9995
    drop_path_rate: float = 0.15
    emb_dim: int = 512
    fusion_layers: int = 2
    fusion_heads: int = 8

    # KD Loss
    kd_alpha: float = 0.7
    kd_temperature: float = 3.0

    # Multi-task
    use_multitask: bool = True
    motion_reg_weight: float = 0.20
    onset_cls_weight: float = 0.15
    severity_cls_weight: float = 0.15

    # Preprocessing
    use_center_crop: bool = True
    use_checkerboard_norm: bool = True
    use_geometry_fold: bool = True
    n_geometry_clusters: int = 16
    use_gem: bool = True
    gem_p_init: float = 3.0
    n_folds: int = 5
    seed: int = 42
    tta_scales: Optional[list] = None

    # Paths
    data_dir: str = '../data'
    output_dir: str = '../outputs'

    def __post_init__(self):
        if self.tta_scales is None:
            self.tta_scales = [self.img_size, self.img_size + 64, self.img_size + 128]


cfg = Config()
seed_everything(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

exp_dir = Path(cfg.output_dir) / cfg.exp_name
exp_dir.mkdir(parents=True, exist_ok=True)
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
print(f'Experiment: {cfg.exp_name}')
print(f'Backbone: {cfg.backbone}')
print(f'Image size: {cfg.img_size}, Batch: {cfg.batch_size}, LR: {cfg.lr}')

Device: cuda
GPU: NVIDIA GeForce RTX 3080
Experiment: v13a_strong_jitter
Backbone: convnextv2_base.fcmae_ft_in22k_in1k
Image size: 384, Batch: 4, LR: 0.0001


---
## ConvNeXt-V2-Base + KD 학습

In [2]:
# === Section 5: Data Loading + Geometry Clustering (v3 reuse + teacher labels) ===

data_dir = Path(cfg.data_dir)
train_df = pd.read_csv(data_dir / 'train.csv')
dev_df = pd.read_csv(data_dir / 'dev.csv')
test_df = pd.read_csv(data_dir / 'sample_submission.csv')

train_df['split'] = 'train'
dev_df['split'] = 'dev'
all_df = pd.concat([train_df, dev_df], ignore_index=True)
all_df['label_int'] = (all_df['label'] == 'unstable').astype(int)
all_df['source_domain'] = all_df['split'].map({'train': 0, 'dev': 1})

# Motion targets (for auxiliary multi-task)
motion_csv = data_dir / 'motion_targets.csv'
if motion_csv.exists():
    motion_df = pd.read_csv(motion_csv)
    all_df = all_df.merge(motion_df, on='id', how='left')

# Teacher soft labels (핵심 변경!)
teacher_csv = data_dir / 'video_teacher_soft_labels.csv'
teacher_df = pd.read_csv(teacher_csv)
all_df = all_df.merge(teacher_df[['id', 'teacher_soft_target']], on='id', how='left')

# soft_target: teacher label for train, hard label for dev
all_df['soft_target'] = all_df['teacher_soft_target'].fillna(all_df['label_int'].astype(float))

print(f'Total: {len(all_df)} (train: {len(train_df)}, dev: {len(dev_df)})')
print(f'Teacher labels available: {all_df["teacher_soft_target"].notna().sum()}')
print(f'Soft target stats (train): mean={all_df.loc[all_df["split"]=="train", "soft_target"].mean():.4f}')

# Geometry clustering (v3 그대로)
def build_geometry_clusters(data_dir, all_df, n_clusters=16):
    data_dir = Path(data_dir)
    front_crop = (96, 80, 288, 320)
    top_crop = (112, 112, 272, 272)
    downsample = (24, 24)
    feats = []
    for _, row in tqdm(all_df.iterrows(), total=len(all_df), desc='geometry-cluster'):
        sid = row['id']
        split_dir = 'train' if row['split'] == 'train' else 'dev'
        base = data_dir / split_dir / sid
        front = cv2.imread(str(base / 'front.png'))
        x1, y1, x2, y2 = front_crop
        front = cv2.cvtColor(front[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
        front = cv2.resize(front, downsample).astype(np.float32) / 255.0
        top = cv2.imread(str(base / 'top.png'))
        x1, y1, x2, y2 = top_crop
        top = cv2.cvtColor(top[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
        top = cv2.resize(top, downsample).astype(np.float32) / 255.0
        feats.append(np.concatenate([front.ravel(), top.ravel()]))
    X = np.stack(feats)
    Xs = StandardScaler().fit_transform(X)
    return KMeans(n_clusters=n_clusters, random_state=42, n_init=20).fit_predict(Xs)

if cfg.use_geometry_fold:
    all_df['geometry_group'] = build_geometry_clusters(cfg.data_dir, all_df, cfg.n_geometry_clusters)
    print(f'Geometry clusters: {all_df["geometry_group"].nunique()} groups')
else:
    all_df['geometry_group'] = 0

Total: 1100 (train: 1000, dev: 100)
Teacher labels available: 1000
Soft target stats (train): mean=0.5216


geometry-cluster:   0%|          | 0/1100 [00:00<?, ?it/s]

geometry-cluster:   1%|          | 11/1100 [00:00<00:10, 104.65it/s]

geometry-cluster:   2%|▏         | 22/1100 [00:00<00:10, 102.75it/s]

geometry-cluster:   3%|▎         | 33/1100 [00:00<00:11, 93.29it/s] 

...

geometry-cluster:  99%|█████████▉| 1088/1100 [00:14<00:00, 50.23it/s]

geometry-cluster:  99%|█████████▉| 1094/1100 [00:14<00:00, 48.54it/s]

geometry-cluster: 100%|█████████▉| 1099/1100 [00:15<00:00, 47.05it/s]

geometry-cluster: 100%|██████████| 1100/1100 [00:15<00:00, 73.03it/s]

Geometry clusters: 16 groups


In [3]:
# === Section 6: Preprocessing + Augmentation (v3 그대로) ===

def center_physics_crop(img, view):
    h, w = img.shape[:2]
    if view == 'front':
        x1, y1 = int(0.25 * w), int(0.20 * h)
        x2, y2 = int(0.75 * w), int(0.88 * h)
    else:
        x1, y1 = int(0.29 * w), int(0.29 * h)
        x2, y2 = int(0.71 * w), int(0.71 * h)
    return img[y1:y2, x1:x2]


def estimate_checkerboard_rotation(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]
    fg_mask = ((sat > 30) | (val < 80) | (val > 220)).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=2)
    bg_mask = cv2.bitwise_not(fg_mask)
    edges = cv2.Canny(gray, 40, 120)
    edges = cv2.bitwise_and(edges, bg_mask)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=30, minLineLength=24, maxLineGap=6)
    if lines is None or len(lines) < 10:
        return None
    angles = []
    for line in lines[:400]:
        x1, y1, x2, y2 = line[0]
        angles.append(np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 90)
    hist, bins = np.histogram(angles, bins=90, range=(0, 90))
    peak_angle = (bins[np.argmax(hist)] + bins[np.argmax(hist) + 1]) / 2
    if hist.max() / (hist.sum() + 1e-6) < 0.08:
        return None
    if peak_angle > 45:
        peak_angle -= 90
    return peak_angle


def normalize_top_rotation(img):
    angle = estimate_checkerboard_rotation(img)
    if angle is None:
        return img
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderValue=(128, 128, 128))


def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=(-0.5, 0.15), contrast_limit=(-0.4, 0.4), p=0.9),
        A.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.1, p=0.8),
        A.GaussianBlur(blur_limit=(3, 5), p=0.35),
        A.RandomGamma(gamma_limit=(60, 140), p=0.5),
        A.Perspective(scale=(0.02, 0.10), p=0.35),
        A.Affine(scale=(0.92, 1.08), translate_percent=(-0.05, 0.05), rotate=(-7, 7), p=0.5),
        A.HorizontalFlip(p=0.5),
        A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(int(img_size*0.02), int(img_size*0.08)),
                        hole_width_range=(int(img_size*0.02), int(img_size*0.08)), fill=0, p=0.10),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], additional_targets={'top': 'image'}, is_check_shapes=False)


def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], additional_targets={'top': 'image'}, is_check_shapes=False)

In [4]:
# === Section 7: Dataset (v3 그대로) ===

class StructuralDatasetV3(Dataset):
    def __init__(self, df, data_dir, transforms=None, is_test=False, cfg=None):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transforms = transforms
        self.is_test = is_test
        self.cfg = cfg or Config()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = row['id']
        split = row.get('split', 'train')
        if self.is_test or split == 'test':
            base = self.data_dir / 'test' / sample_id
        elif split == 'dev':
            base = self.data_dir / 'dev' / sample_id
        else:
            base = self.data_dir / 'train' / sample_id

        front = cv2.cvtColor(cv2.imread(str(base / 'front.png')), cv2.COLOR_BGR2RGB)
        top = cv2.cvtColor(cv2.imread(str(base / 'top.png')), cv2.COLOR_BGR2RGB)

        if self.cfg.use_center_crop:
            front = center_physics_crop(front, 'front')
            top = center_physics_crop(top, 'top')
        if self.cfg.use_checkerboard_norm:
            top = normalize_top_rotation(top)

        if self.transforms:
            augmented = self.transforms(image=front, top=top)
            front = augmented['image']
            top = augmented['top']

        result = {'front': front, 'top': top, 'id': sample_id}
        if not self.is_test:
            result['label'] = int(row['label_int'])
            result['soft_target'] = float(row.get('soft_target', row['label_int']))
            result['max_diff_first'] = float(row['max_diff_first']) if pd.notna(row.get('max_diff_first')) else -1.0
            result['mean_diff_prev'] = float(row['mean_diff_prev']) if pd.notna(row.get('mean_diff_prev')) else -1.0
            result['onset_bucket'] = int(row['onset_bucket']) if pd.notna(row.get('onset_bucket')) else -1
            result['severity_bucket'] = int(row['severity_bucket']) if pd.notna(row.get('severity_bucket')) else -1
        return result

In [5]:
# === Section 8: Model (v3 그대로) ===

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p).flatten(1)


class DualStreamModelV3(nn.Module):
    def __init__(self, backbone_name, emb_dim=512, drop_path_rate=0.15,
                 use_gem=True, gem_p=3.0, fusion_layers=2, fusion_heads=8):
        super().__init__()
        self.emb_dim = emb_dim
        self.backbone_front = timm.create_model(backbone_name, pretrained=True, num_classes=0, drop_path_rate=drop_path_rate)
        self.backbone_top = timm.create_model(backbone_name, pretrained=True, num_classes=0, drop_path_rate=drop_path_rate)
        for bb in [self.backbone_front, self.backbone_top]:
            if hasattr(bb, 'set_grad_checkpointing'):
                bb.set_grad_checkpointing(True)
        feat_dim = self.backbone_front.num_features
        self.gem_front = GeM(p=gem_p) if use_gem else nn.AdaptiveAvgPool2d(1)
        self.gem_top = GeM(p=gem_p) if use_gem else nn.AdaptiveAvgPool2d(1)
        self.proj_front = nn.Sequential(nn.Linear(feat_dim, emb_dim), nn.GELU(), nn.Dropout(0.15))
        self.proj_top = nn.Sequential(nn.Linear(feat_dim, emb_dim), nn.GELU(), nn.Dropout(0.15))
        self.view_embed = nn.Parameter(torch.randn(2, emb_dim) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=fusion_heads, dim_feedforward=emb_dim * 4,
            dropout=0.10, batch_first=True, activation='gelu', norm_first=True)
        self.fusion = nn.TransformerEncoder(encoder_layer, num_layers=fusion_layers)
        self.norm = nn.LayerNorm(emb_dim)
        fused_dim = emb_dim * 3
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim), nn.Linear(fused_dim, emb_dim), nn.GELU(), nn.Dropout(0.20), nn.Linear(emb_dim, 1))
        self.motion_head = nn.Sequential(nn.Linear(fused_dim, emb_dim // 2), nn.GELU(), nn.Linear(emb_dim // 2, 2))
        self.onset_head = nn.Sequential(nn.Linear(fused_dim, emb_dim // 2), nn.GELU(), nn.Linear(emb_dim // 2, 4))
        self.severity_head = nn.Sequential(nn.Linear(fused_dim, emb_dim // 2), nn.GELU(), nn.Linear(emb_dim // 2, 4))

    def forward(self, front, top):
        f_feat = self.backbone_front(front)
        t_feat = self.backbone_top(top)
        if f_feat.ndim > 2:
            f_feat = self.gem_front(f_feat).flatten(1)
            t_feat = self.gem_top(t_feat).flatten(1)
        f_emb = self.proj_front(f_feat)
        t_emb = self.proj_top(t_feat)
        tokens = torch.stack([f_emb + self.view_embed[0], t_emb + self.view_embed[1]], dim=1)
        fused = self.fusion(tokens)
        fused_mean = self.norm(fused.mean(dim=1))
        feat = torch.cat([f_emb, t_emb, fused_mean], dim=1)
        return {
            'logit': self.classifier(feat).squeeze(1),
            'motion_reg': self.motion_head(feat),
            'onset_logit': self.onset_head(feat),
            'severity_logit': self.severity_head(feat),
        }

In [6]:
# === Section 9: Loss (KD) + EMA + Scheduler + Temperature Scaling ===

class ModelEmaV2(nn.Module):
    def __init__(self, model, decay=0.9995):
        super().__init__()
        self.module = copy.deepcopy(model).cpu()
        self.module.eval()
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.module.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data.cpu(), alpha=1.0 - self.decay)


class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, steps_per_epoch):
        self.optimizer = optimizer
        self.warmup_steps = warmup_epochs * steps_per_epoch
        self.total_steps = total_epochs * steps_per_epoch
        self.current_step = 0
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            scale = self.current_step / max(self.warmup_steps, 1)
        else:
            progress = (self.current_step - self.warmup_steps) / max(self.total_steps - self.warmup_steps, 1)
            scale = 0.5 * (1 + math.cos(math.pi * progress))
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * scale


def compute_loss_kd(outputs, batch, cfg):
    """KD Loss: alpha * BCE(logit, teacher_soft) + (1-alpha) * BCE(logit, hard_label) + aux"""
    logit = outputs['logit']
    hard_label = batch['label'].float().to(logit.device)
    teacher_soft = batch['soft_target'].float().to(logit.device)

    kd_loss = F.binary_cross_entropy_with_logits(logit, teacher_soft)
    hard_loss = F.binary_cross_entropy_with_logits(logit, hard_label)
    main_loss = cfg.kd_alpha * kd_loss + (1 - cfg.kd_alpha) * hard_loss

    total_loss = main_loss

    if cfg.use_multitask:
        max_df = batch['max_diff_first'].float().to(logit.device)
        mean_dp = batch['mean_diff_prev'].float().to(logit.device)
        valid_motion = max_df >= 0
        if valid_motion.any():
            motion_tgt = torch.stack([max_df[valid_motion] / 10.0, mean_dp[valid_motion] / 0.15], dim=1).clamp(0, 2)
            total_loss = total_loss + cfg.motion_reg_weight * F.smooth_l1_loss(outputs['motion_reg'][valid_motion], motion_tgt)

        onset = batch['onset_bucket'].long().to(logit.device)
        valid_onset = onset >= 0
        if valid_onset.any():
            total_loss = total_loss + cfg.onset_cls_weight * F.cross_entropy(outputs['onset_logit'][valid_onset], onset[valid_onset])

        sev = batch['severity_bucket'].long().to(logit.device)
        valid_sev = sev >= 0
        if valid_sev.any():
            total_loss = total_loss + cfg.severity_cls_weight * F.cross_entropy(outputs['severity_logit'][valid_sev], sev[valid_sev])

    return total_loss


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def fit(self, logits, y_true, max_iter=200):
        dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.to(dev)
        x = torch.tensor(logits, dtype=torch.float32, device=dev)
        y = torch.tensor(y_true, dtype=torch.float32, device=dev)
        opt = torch.optim.LBFGS(self.parameters(), lr=0.1, max_iter=max_iter)
        def closure():
            opt.zero_grad(set_to_none=True)
            loss = F.binary_cross_entropy_with_logits(x / self.temperature, y)
            loss.backward()
            return loss
        opt.step(closure)
        temp = float(self.temperature.detach().cpu().item())
        return max(temp, 0.01)


print('KD Loss, EMA, Scheduler, Temperature Scaler defined.')

KD Loss, EMA, Scheduler, Temperature Scaler defined.


In [7]:
# === Section 10: Train Loop (v3 그대로, loss만 KD로 교체) ===

def train_one_phase(model, train_loader, val_loader, optimizer, scheduler, cfg, phase_epochs, ema_model=None):
    scaler = torch.amp.GradScaler('cuda')
    best_val_loss = float('inf')
    best_state = None
    best_ema_state = None
    best_ema_loss = float('inf')
    patience_counter = 0
    val_labels = None
    val_logits_for_temp = None

    for epoch in range(phase_epochs):
        model.train()
        running_loss = 0.0
        step_count = 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{phase_epochs} [Train]', leave=False)
        for batch in pbar:
            front = batch['front'].to(device)
            top = batch['top'].to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda', dtype=torch.bfloat16):
                outputs = model(front, top)
                loss = compute_loss_kd(outputs, batch, cfg)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            if ema_model is not None:
                ema_model.update(model)
            running_loss += loss.item()
            step_count += 1
            pbar.set_postfix(loss=f'{running_loss / step_count:.4f}')

        # Validation
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]', leave=False):
                front = batch['front'].to(device)
                top = batch['top'].to(device)
                with autocast('cuda', dtype=torch.bfloat16):
                    outputs = model(front, top)
                all_logits.append(outputs['logit'].float().cpu())
                all_labels.append(batch['label'])
        all_logits = torch.cat(all_logits).numpy()
        all_labels = torch.cat(all_labels).numpy()
        val_probs = sigmoid_np(all_logits)
        val_loss = log_loss(all_labels, val_probs, labels=[0, 1])
        val_auc = roc_auc_score(all_labels, val_probs)
        print(f'Epoch {epoch+1}: val_loss={val_loss:.4f}, val_auc={val_auc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            val_labels = all_labels
            val_logits_for_temp = all_logits
            patience_counter = 0
        else:
            patience_counter += 1

        if ema_model is not None:
            ema_model.module.to(device)
            ema_model.module.eval()
            ema_logits = []
            with torch.no_grad():
                for batch in val_loader:
                    front = batch['front'].to(device)
                    top = batch['top'].to(device)
                    with autocast('cuda', dtype=torch.bfloat16):
                        out = ema_model.module(front, top)
                    ema_logits.append(out['logit'].float().cpu())
            ema_model.module.cpu()
            ema_logits = torch.cat(ema_logits).numpy()
            ema_probs = sigmoid_np(ema_logits)
            ema_loss = log_loss(all_labels, ema_probs, labels=[0, 1])
            print(f'  EMA val_loss={ema_loss:.4f}')
            if ema_loss < best_ema_loss:
                best_ema_loss = ema_loss
                best_ema_state = {k: v.clone() for k, v in ema_model.module.state_dict().items()}

        if patience_counter >= cfg.early_stopping_patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

    return best_state, best_val_loss, val_labels, val_logits_for_temp, best_ema_state, best_ema_loss

In [8]:
# === Section 11: 5-Fold CV ===

def train_one_fold(fold, train_idx, val_idx, all_df, cfg):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}')
    print(f'{"="*60}')
    train_data = all_df.iloc[train_idx]
    val_data = all_df.iloc[val_idx]
    print(f'Train: {len(train_data)} | Val: {len(val_data)}')

    fold_dir = exp_dir / f'fold{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)

    model = DualStreamModelV3(
        cfg.backbone, emb_dim=cfg.emb_dim, drop_path_rate=cfg.drop_path_rate,
        use_gem=cfg.use_gem, gem_p=cfg.gem_p_init,
        fusion_layers=cfg.fusion_layers, fusion_heads=cfg.fusion_heads
    ).to(device)

    train_ds = StructuralDatasetV3(train_data, data_dir, get_train_transforms(cfg.img_size), cfg=cfg)
    val_ds = StructuralDatasetV3(val_data, data_dir, get_val_transforms(cfg.img_size), cfg=cfg)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0, pin_memory=True)

    backbone_params = list(model.backbone_front.parameters()) + list(model.backbone_top.parameters())
    head_params = [p for n, p in model.named_parameters() if 'backbone' not in n]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': cfg.lr * 0.5},
        {'params': head_params, 'lr': cfg.lr},
    ], weight_decay=cfg.weight_decay)
    scheduler = CosineWarmupScheduler(optimizer, cfg.warmup_epochs, cfg.epochs, len(train_loader))
    ema_model = ModelEmaV2(model, decay=cfg.ema_decay) if cfg.use_ema else None

    best_state, best_loss, val_labels, val_logits, best_ema_state, best_ema_loss = train_one_phase(
        model, train_loader, val_loader, optimizer, scheduler, cfg, cfg.epochs, ema_model=ema_model)

    if cfg.use_ema and best_ema_state is not None and best_ema_loss < best_loss:
        best_state = best_ema_state
        best_loss = best_ema_loss
        print(f'Using EMA model (loss={best_ema_loss:.4f})')

    temp = 1.0
    if val_logits is not None:
        ts = TemperatureScaler()
        temp = ts.fit(val_logits, val_labels)
        cal_probs = sigmoid_np(val_logits / temp)
        cal_loss = log_loss(val_labels, cal_probs, labels=[0, 1])
        print(f'Temperature: {temp:.4f}, Calibrated loss: {cal_loss:.4f}')
        del ts

    torch.save(best_state, fold_dir / 'best_model.pt')
    with open(fold_dir / 'temperature.json', 'w') as f:
        json.dump({'temperature': temp, 'val_logloss': float(best_loss)}, f)

    model.load_state_dict(best_state)
    model.to(device)
    model.eval()
    oof_logits = []
    with torch.no_grad():
        for batch in val_loader:
            front = batch['front'].to(device)
            top = batch['top'].to(device)
            with autocast('cuda', dtype=torch.bfloat16):
                out = model(front, top)
            oof_logits.append(out['logit'].float().cpu())
    oof_logits = torch.cat(oof_logits).numpy()
    oof_probs = sigmoid_np(oof_logits / temp)
    oof_preds = np.stack([1 - oof_probs, oof_probs], axis=1)
    oof_logloss = log_loss(val_labels, oof_preds, labels=[0, 1])
    print(f'Fold {fold} Final OOF LogLoss: {oof_logloss:.4f}')

    del model
    torch.cuda.empty_cache()
    return oof_preds, val_idx, oof_logloss, temp


# Run
y_strat = all_df['label_int'].astype(str) + '_' + all_df['source_domain'].astype(str)
groups = all_df['geometry_group'].values
skf = StratifiedGroupKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
splits = list(skf.split(all_df, y_strat, groups))

oof_predictions = np.zeros((len(all_df), 2))
fold_scores = []
fold_temps = []

for fold, (train_idx, val_idx) in enumerate(splits):
    oof_preds, val_idx_out, fold_score, temp = train_one_fold(fold, train_idx, val_idx, all_df, cfg)
    oof_predictions[val_idx_out] = oof_preds
    fold_scores.append(fold_score)
    fold_temps.append(temp)

overall_logloss = log_loss(all_df['label_int'].values, oof_predictions, labels=[0, 1])
overall_auc = roc_auc_score(all_df['label_int'].values, oof_predictions[:, 1])

print(f'\n{"="*60}')
print(f'OVERALL CV RESULTS ({cfg.exp_name})')
print(f'{"="*60}')
for i, score in enumerate(fold_scores):
    print(f'  Fold {i}: LogLoss = {score:.4f}')
print(f'  Mean:   LogLoss = {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')
print(f'  Overall LogLoss = {overall_logloss:.4f}')
print(f'  Overall AUC     = {overall_auc:.4f}')
np.save(exp_dir / 'oof_preds.npy', oof_predictions)


FOLD 0
Train: 871 | Val: 229


Epoch 1/15 [Train]:   0%|          | 0/217 [00:00<?, ?it/s]

Epoch 1/15 [Train]:   0%|          | 0/217 [00:06<?, ?it/s, loss=1.1186]

Epoch 1/15 [Train]:   0%|          | 1/217 [00:06<23:55,  6.65s/it, loss=1.1186]

Epoch 1/15 [Train]:   0%|          | 1/217 [00:07<23:55,  6.65s/it, loss=1.2197]

...

Epoch 8 [Val]: 100%|██████████| 30/30 [00:05<00:00,  5.62it/s]

Epoch 8: val_loss=0.6913, val_auc=0.4884


  EMA val_loss=0.6928
Early stopping at epoch 8
Using EMA model (loss=0.3673)


Temperature: 1.8712, Calibrated loss: 0.4524


Fold 4 Final OOF LogLoss: 0.4944

OVERALL CV RESULTS (v13a_strong_jitter)
  Fold 0: LogLoss = 0.0000
  Fold 1: LogLoss = 0.0000
  Fold 2: LogLoss = 0.0909
  Fold 3: LogLoss = 0.0000
  Fold 4: LogLoss = 0.4944
  Mean:   LogLoss = 0.1171 +/- 0.1919
  Overall LogLoss = 0.1260
  Overall AUC     = 0.9981


In [9]:
# === Section 12: Test Inference + TTA ===

def predict_test(cfg):
    test_df_local = pd.read_csv(Path(cfg.data_dir) / 'sample_submission.csv')
    test_df_local['split'] = 'test'
    all_fold_preds = []

    for fold in range(cfg.n_folds):
        fold_dir = exp_dir / f'fold{fold}'
        print(f'\nFold {fold} inference...')
        model = DualStreamModelV3(
            cfg.backbone, emb_dim=cfg.emb_dim, drop_path_rate=0,
            use_gem=cfg.use_gem, gem_p=cfg.gem_p_init,
            fusion_layers=cfg.fusion_layers, fusion_heads=cfg.fusion_heads
        ).to(device)
        model.load_state_dict(torch.load(fold_dir / 'best_model.pt', weights_only=True))
        model.eval()
        with open(fold_dir / 'temperature.json') as f:
            temp = json.load(f)['temperature']

        fold_tta_preds = []
        for scale in cfg.tta_scales:
            for flip in [False, True]:
                transforms = get_val_transforms(scale)
                test_ds = StructuralDatasetV3(test_df_local, Path(cfg.data_dir), transforms, is_test=True, cfg=cfg)
                test_loader = DataLoader(test_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0, pin_memory=True)
                tta_logits = []
                with torch.no_grad():
                    for batch in test_loader:
                        front = batch['front'].to(device)
                        top = batch['top'].to(device)
                        if flip:
                            front = torch.flip(front, dims=[3])
                            top = torch.flip(top, dims=[3])
                        with autocast('cuda', dtype=torch.bfloat16):
                            out = model(front, top)
                        tta_logits.append(out['logit'].float().cpu())
                tta_logits = torch.cat(tta_logits).numpy()
                fold_tta_preds.append(sigmoid_np(tta_logits / temp))

        fold_mean = np.mean(fold_tta_preds, axis=0)
        all_fold_preds.append(fold_mean)
        print(f'  Fold {fold}: mean pred = {fold_mean.mean():.4f}')
        del model
        torch.cuda.empty_cache()

    return np.mean(all_fold_preds, axis=0)


test_preds = predict_test(cfg)
np.save(exp_dir / 'test_preds.npy', test_preds)
print(f'Test predictions: shape={test_preds.shape}, mean={test_preds.mean():.4f}')


Fold 0 inference...


  Fold 0: mean pred = 0.6551

Fold 1 inference...


  Fold 1: mean pred = 0.4921

Fold 2 inference...


  Fold 2: mean pred = 0.5283

Fold 3 inference...


  Fold 3: mean pred = 0.4918

Fold 4 inference...


  Fold 4: mean pred = 0.4403
Test predictions: shape=(1000,), mean=0.5215


In [10]:
# === Section 13: Submission ===

test_df_sub = pd.read_csv(Path(cfg.data_dir) / 'sample_submission.csv')
unstable_prob = np.clip(test_preds, 1e-6, 1 - 1e-6)
test_df_sub['unstable_prob'] = unstable_prob
test_df_sub['stable_prob'] = 1.0 - unstable_prob

submissions_dir = Path('../submissions')
submissions_dir.mkdir(parents=True, exist_ok=True)
submission_path = submissions_dir / f'{cfg.exp_name}_submission.csv'
test_df_sub[['id', 'unstable_prob', 'stable_prob']].to_csv(submission_path, index=False)

print(f'\n{"="*60}')
print(f'SUBMISSION: {submission_path}')
print(f'{"="*60}')
print(f'Samples: {len(test_df_sub)}')
print(f'Unstable mean: {unstable_prob.mean():.4f}')
print(f'CV LogLoss: {overall_logloss:.4f}')
print(f'CV AUC: {overall_auc:.4f}')


SUBMISSION: ..\submissions\v13a_strong_jitter_submission.csv
Samples: 1000
Unstable mean: 0.5215
CV LogLoss: 0.1260
CV AUC: 0.9981
